###  import  and  get   the  data sets    

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/crop_yield.csv")   

print("Shape:", df.shape)
df.head()    

Shape: (1000000, 10)


,Region,Soil_Type,Crop,Rainfall_mm,Temperature_Celsius,Fertilizer_Used,Irrigation_Used,Weather_Condition,Days_to_Harvest,Yield_tons_per_hectare
0,West,Sandy,Cotton,897.077239,27.676966,False,True,Cloudy,122,6.555816
1,South,Clay,Rice,992.673282,18.026142,True,True,Rainy,140,8.527341
2,North,Loam,Barley,147.998025,29.794042,False,False,Sunny,106,1.127443
3,North,Sandy,Soybean,986.866331,16.644190,False,True,Rainy,146,6.517573
4,South,Silt,Wheat,730.379174,31.620687,True,True,Cloudy,110,7.248251


In [2]:
### data  types  and   information    
df.info()  

<class 'pandas.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 10 columns):
 #   Column                  Non-Null Count    Dtype  
---  ------                  --------------    -----  
 0   Region                  1000000 non-null  str    
 1   Soil_Type               1000000 non-null  str    
 2   Crop                    1000000 non-null  str    
 3   Rainfall_mm             1000000 non-null  float64
 4   Temperature_Celsius     1000000 non-null  float64
 5   Fertilizer_Used         1000000 non-null  bool   
 6   Irrigation_Used         1000000 non-null  bool   
 7   Weather_Condition       1000000 non-null  str    
 8   Days_to_Harvest         1000000 non-null  int64  
 9   Yield_tons_per_hectare  1000000 non-null  float64
dtypes: bool(2), float64(3), int64(1), str(4)
memory usage: 62.9 MB


In [3]:
df.describe()    

,Rainfall_mm,Temperature_Celsius,Days_to_Harvest,Yield_tons_per_hectare
count,1000000.000000,1000000.000000,1000000.000000,1000000.000000
mean,549.981901,27.504965,104.495025,4.649472
std,259.851320,7.220608,25.953412,1.696572
min,100.000896,15.000034,60.000000,-1.147613
25%,324.891090,21.254502,82.000000,3.417637
50%,550.124061,27.507365,104.000000,4.651808
75%,774.738520,33.753267,127.000000,5.879200
max,999.998098,39.999997,149.000000,9.963372


In [4]:
###  get   the  categoraical  informations  
df.describe(include='object')   

C:\Users\ASUS\AppData\Local\Temp\ipykernel_15368\4067014382.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.describe(include='object')


,Region,Soil_Type,Crop,Weather_Condition
count,1000000,1000000,1000000,1000000
unique,4,6,6,3
top,North,Sandy,Maize,Sunny
freq,250173,167119,166824,333790


###  check   the  missing  values   

In [5]:
missing = df.isnull().sum()
missing_percent = (missing / len(df)) * 100

missing_df = pd.DataFrame({"Missing Count": missing, "Missing %": missing_percent})
missing_df[missing_df["Missing Count"] > 0].sort_values("Missing %", ascending=False)    

,Missing Count,Missing %


###  check   the  duplications    


In [6]:
duplicate_count = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicate_count}")   

Number of duplicate rows: 0


In [7]:
categorical_cols = ["Region", "Soil_Type", "Crop", "Weather_Condition"]

for col in categorical_cols:
    print(f"\n{col} — {df[col].nunique()} unique values")
    print(df[col].unique())


Region — 4 unique values
<StringArray>
['West', 'South', 'North', 'East']
Length: 4, dtype: str

Soil_Type — 6 unique values
<StringArray>
['Sandy', 'Clay', 'Loam', 'Silt', 'Peaty', 'Chalky']
Length: 6, dtype: str

Crop — 6 unique values
<StringArray>
['Cotton', 'Rice', 'Barley', 'Soybean', 'Wheat', 'Maize']
Length: 6, dtype: str

Weather_Condition — 3 unique values
<StringArray>
['Cloudy', 'Rainy', 'Sunny']
Length: 3, dtype: str


In [8]:
###  boolean columns  check   
print("Fertilizer_Used values:", df["Fertilizer_Used"].unique())
print("Irrigation_Used values:", df["Irrigation_Used"].unique())
print()
print(df["Fertilizer_Used"].value_counts())
print()
print(df["Irrigation_Used"].value_counts())   

Fertilizer_Used values: [False  True]
Irrigation_Used values: [ True False]

Fertilizer_Used
False    500060
True     499940
Name: count, dtype: int64

Irrigation_Used
False    500509
True     499491
Name: count, dtype: int64


###  target  variable distributions  

In [9]:
target_col = "Yield_tons_per_hectare"

print(df[target_col].describe())
print("\nSkewness:", df[target_col].skew())  


count    1000000.000000
mean           4.649472
std            1.696572
min           -1.147613
25%            3.417637
50%            4.651808
75%            5.879200
max            9.963372
Name: Yield_tons_per_hectare, dtype: float64

Skewness: -0.0008624714646197753


In [10]:
###  Quick correlation/signal check    
from sklearn.preprocessing import LabelEncoder

check_df = df.copy()
for col in categorical_cols:
    check_df[col] = LabelEncoder().fit_transform(check_df[col])
check_df["Fertilizer_Used"] = check_df["Fertilizer_Used"].astype(int)
check_df["Irrigation_Used"] = check_df["Irrigation_Used"].astype(int)

correlations = check_df.corr(numeric_only=True)[target_col].drop(target_col).sort_values(key=abs, ascending=False)
correlations

Rainfall_mm            0.764618
Fertilizer_Used        0.442099
Irrigation_Used        0.353741
Temperature_Celsius    0.085565
Days_to_Harvest       -0.002591
Crop                   0.001283
Weather_Condition      0.001132
Region                 0.000390
Soil_Type             -0.000333
Name: Yield_tons_per_hectare, dtype: float64

###  save  the  summary  of  the  data understaning    


In [11]:
import json

summary = {
    "n_rows": df.shape[0],
    "n_columns": df.shape[1],
    "columns": df.columns.tolist(),
    "categorical_columns": categorical_cols,
    "boolean_columns": ["Fertilizer_Used", "Irrigation_Used"],
    "numerical_columns": ["Rainfall_mm", "Temperature_Celsius", "Days_to_Harvest"],
    "target_column": target_col,
    "missing_values": int(missing.sum()),
    "duplicate_rows": int(duplicate_count),
    "top_correlated_feature": correlations.index[0],
    "top_correlation_value": float(correlations.iloc[0])
}

with open("../reports/data_understanding_summary.json", "w") as f:
    json.dump(summary, f, indent=4)

summary   

{'n_rows': 1000000,
 'n_columns': 10,
 'columns': ['Region',
  'Soil_Type',
  'Crop',
  'Rainfall_mm',
  'Temperature_Celsius',
  'Fertilizer_Used',
  'Irrigation_Used',
  'Weather_Condition',
  'Days_to_Harvest',
  'Yield_tons_per_hectare'],
 'categorical_columns': ['Region', 'Soil_Type', 'Crop', 'Weather_Condition'],
 'boolean_columns': ['Fertilizer_Used', 'Irrigation_Used'],
 'numerical_columns': ['Rainfall_mm',
  'Temperature_Celsius',
  'Days_to_Harvest'],
 'target_column': 'Yield_tons_per_hectare',
 'missing_values': 0,
 'duplicate_rows': 0,
 'top_correlated_feature': 'Rainfall_mm',
 'top_correlation_value': 0.7646179592717693}